#### 연습 문제 
- data 폴더 안에 가전 폴더의 모든 json 파일을 하나의 데이터프레임으로 단순 행 결합 
- 데이터프레임의 정보를 확인 
    - 컬럼들의 타입 
    - 결측치의 확인 
- 결측치가 포함되어 있는 데이터를 추출하여 따로 저장 (df_na)
- 원본데이터에서는 결측치를 제외
- 'RawText', 'GeneralPolarity' 를 제외하고 나머지는 제거 
- RawText 컬럼은 텍스트 정규화(특수 문자 제거, 2칸 이상의 공백 처리, 문자열 앞뒤 공백 제거)
    - 글자수가 1이하인 데이터 제외
- GeneralPolarity의 데이터는 -1(부정), 0(중립), 1(긍정)
    - 선형 모델에서 출력 차원의 수를 이용하여 높은 위치가 분류 값으로 사용이 되기 때문에 부정을 0으로 중립을 1 긍정을 2로 변환
    - 해당 컬럼의 타입을 int 변경 
- 데이터를 상위 1000개만 추출 
- train, test로 데이터를 분할 
    - 비율은 5:5
    - 계층화 작업 
- 모델은 beomi/kcbert-base 사용
- AutoTokenizer를 이용하여 토큰화 함수 로드 (max_length= 128)
- 같은 모델을 로드하여 BertModel + Linear 모델을 정의 
- 평가 지표 함수를 생성 
    - f1-score는 average = 'macro' 사용
- Trainer, TrainingArguments를 이용하여 학습 관리 객체 생성 
- train() 함수를 이용하여 파인 튜닝 (3회)
- 검증 작업 

In [1]:
# 텍스트 정규화
import re 
# 특정 폴더의 파일의 목록을 불러오기 위한 라이브러리 
import os 
from glob import glob
import pandas as pd 
import torch 
import torch.nn as nn 
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments

In [2]:
MODEL_NAME = 'beomi/kcbert-base'

In [ ]:
# 특정 경로의 파일 목록 불러오기 
file_path = "../data/가전/"
file_list = os.listdir(file_path)
pd.read_json(file_path + file_list[0])

In [ ]:
file_list2 = glob("../data/가전/*.json")
pd.read_json(file_list2[0])

In [9]:
# 특정 폴더의 데이터들을 DataFrame으로 생성하여 하나의 데이터프레임으로 결합 
df = pd.DataFrame()
for file_name in file_list2:
    data = pd.read_json(file_name)
    # 단순 결합 
    df = pd.concat([df, data], axis=0)
df.reset_index(drop=True, inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4056 entries, 0 to 4055
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   str    
 2   Source           4056 non-null   str    
 3   Domain           4056 non-null   str    
 4   MainCategory     4056 non-null   str    
 5   ProductName      4056 non-null   str    
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 3.9+ MB


In [10]:
pd.concat(
    [ pd.read_json(file_name) for file_name in file_list2 ]
).info()

<class 'pandas.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   str    
 2   Source           4056 non-null   str    
 3   Domain           4056 non-null   str    
 4   MainCategory     4056 non-null   str    
 5   ProductName      4056 non-null   str    
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 3.9+ MB


In [11]:
pd.concat(
    list(
        map(
            lambda x : pd.read_json(x), 
            file_list2
        )
    )
).info()

<class 'pandas.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   str    
 2   Source           4056 non-null   str    
 3   Domain           4056 non-null   str    
 4   MainCategory     4056 non-null   str    
 5   ProductName      4056 non-null   str    
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 3.9+ MB


In [12]:
# 결측치의 개수 확인 
df.isna().sum()

Index                0
RawText              0
Source               0
Domain               0
MainCategory         0
ProductName          0
ReviewScore          0
Syllable             0
Word                 0
RDate                0
GeneralPolarity    378
Aspects              0
dtype: int64

In [ ]:
# Polarity 컬럼의 결측치는 따로 추출하여 예측에서 사용할 데이터로 저장 
df_na = df.loc[
    df['GeneralPolarity'].isna(), 
]
df_na.head()

In [14]:
# 원본의 df는 결측치를 제거 
df.dropna(inplace=True)

In [35]:
# 데이터의 개수를 1000만 사용 
df2 = df[:1000]
df2['GeneralPolarity'].value_counts()

GeneralPolarity
 1.0    565
 0.0    260
-1.0    175
Name: count, dtype: int64

In [36]:
# 특정 컬러만 사용 
df2 = df2[ ['RawText', 'GeneralPolarity'] ]

In [37]:
df2.columns

Index(['RawText', 'GeneralPolarity'], dtype='str')

In [38]:
# 텍스트 정규화 
# GeneralPolarity의 값을 int로 변경하고 0, 1, 2 변경 
def normalize(data):
    # data : df2의 인덱스 하나씩 대입(스리즈) 
    data['RawText'] = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", ' ', data['RawText'])
    data['RawText'] = re.sub(r"\s+", ' ', data['RawText'])

    data['GeneralPolarity'] = int(data['GeneralPolarity'])
    data['GeneralPolarity'] += 1
    # data['GenenalPolarity'] = data['GenenalPolarity'].map(
    #     {
    #         -1 : 0, 
    #         0 : 1, 
    #         1 : 2
    #     }
    # )
    return data

In [39]:
df3 = df2.apply(normalize, axis=1)

In [40]:
# df2.apply(lambda x : print(x), axis=1)

In [44]:
# train, test 데이터 분할
train_df, test_df = train_test_split(
    df3, test_size= 0.5, random_state=42, stratify=df3['GeneralPolarity']
)

In [45]:
train_df['GeneralPolarity'].value_counts()

GeneralPolarity
2    283
1    130
0     87
Name: count, dtype: int64

In [46]:
# Dataset으로 생성 
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

In [47]:
train_ds

Dataset({
    features: ['RawText', 'GeneralPolarity'],
    num_rows: 500
})

In [48]:
# 토크나이저 로드 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast = False)

In [49]:
def token_fn(batch):
    result = tokenizer(
        batch['RawText'], 
        truncation = True, 
        max_length = 128, 
        padding = True, 
        return_tensor = 'pt'    
    )
    return result

train_tok = train_ds.map(token_fn, batched=True, remove_columns=['RawText'])
test_tok = test_ds.map(token_fn, batched=True, remove_columns=['RawText'])

Map: 100%|██████████| 500/500 [00:00<00:00, 6871.31 examples/s]


In [50]:
train_tok

Dataset({
    features: ['GeneralPolarity', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})

In [88]:
# BERT + Linear 학습 모델을 선언 
class BERTCLF(nn.Module):
    def __init__(self, model_name, num_claases = 2, dropout = 0.5):
        super().__init__()
        # 기존에 학습 된 모델을 로드 
        self.backbone = BertModel.from_pretrained(model_name)

        # 로드한 모델의 출력 차원의 수 
        hidden = self.backbone.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, num_claases)
        # 작업의 안정성을 위해서 토크나이저의 패딩 토큰 아이디를 로드한 모델에 패딩 토큰 아이디로 사용
        self.backbone.config.pad_token_id = tokenizer.pad_token_id
    
    def forward(self, input_ids = None, attention_mask = None, 
                GeneralPolarity = None, **kwargs):
        out = self.backbone(input_ids = input_ids, attnetion_mask = attention_mask)

        # CLS 부분만 추출
        pooled = out.last_hidden_state[:, 0]
        # 일부 데이터 소실(과적합 방지용)
        drop_data = self.dropout(pooled)

        logits = self.fc(drop_data)

        result = {
            "Logits" : logits
        }

        if GeneralPolarity is not None:
            loss = nn.CrossEntropyLoss()(logits, GeneralPolarity)
            result['loss'] = loss
        
        return result


In [89]:
train_tok

Dataset({
    features: ['GeneralPolarity', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})

In [90]:
# **kwargs 매개변수의 의미 
def test_forward(input_ids = None, attention_mask = None, labels = None, **kwargs):
    print(input_ids[0])
    print(attention_mask[0])
    print(labels)

In [91]:
test_forward(input_ids = train_tok[0]['input_ids'], attention_mask=train_tok[0]['attention_mask'], 
             labels = train_tok[0]['GeneralPolarity'])

2
1
2


In [92]:
train_tok[0]
# 매개변수에 * -> 여러개의 인자값들을 하나의 변수에 저장 
# 인자에 * -> 1차원 데이터의 각 원소들을 각각의 인자로 사용
# 매개변수에 ** -> 함수에서 생성하지 않은 매개변수가 들어올때 사용
# 인자에 ** -> 딕셔너리형 데이터를 key값을 매개변수명으로 사용하고 value를 인자로 사용하여 함수를 호출 

{'GeneralPolarity': 2,
 'input_ids': [2,
  12208,
  4113,
  26130,
  10868,
  4017,
  8381,
  8124,
  10915,
  9082,
  9335,
  9021,
  8013,
  8534,
  18849,
  10868,
  4113,
  15091,
  8159,
  13054,
  248,
  15186,
  2005,
  14893,
  8472,
  13145,
  11514,
  13419,
  17,
  12208,
  10868,
  4029,
  8066,
  16505,
  4042,
  25195,
  7993,
  14184,
  9119,
  9747,
  4007,
  10267,
  9921,
  4061,
  8039,
  13776,
  963,
  4334,
  16505,
  4091,
  11514,
  8518,
  12208,
  4083,
  8823,
  2417,
  4158,
  17,
  3087,
  8383,
  21,
  17,
  27010,
  4008,
  3087,
  8025,
  17,
  17938,
  20277,
  4017,
  13492,
  832,
  3010,
  16707,
  8260,
  23220,
  17,
  8547,
  3089,
  2492,
  13351,
  3453,
  4098,
  4128,
  1371,
  5041,
  10943,
  13576,
  1072,
  4414,
  7968,
  10383,
  16798,
  11429,
  8262,
  4153,
  2535,
  8074,
  17,
  8108,
  1279,
  4137,
  5233,
  4017,
  9855,
  23239,
  4047,
  16612,
  4017,
  8545,
  4032,
  3294,
  4598,
  4103,
  16612,
  4017,
  2483,
  10009,
 

In [93]:
test_forward(**train_tok[0])

2
1
None


In [94]:
test_dict = {
    'input_ids' : [10, 20, 30], 
    'attention_mask' : [1, 1, 1], 
    'labels' : 2, 
    'GeneralPolarity' : 1
}

In [95]:
test_forward(**test_dict)

10
1
2


In [96]:
test_forward(input_ids=test_dict['input_ids'], attention_mask=test_dict['attention_mask'], 
             labels=test_dict['labels'], GeneralPolarity = test_dict['GeneralPolarity'])

10
1
2


In [97]:
# 모델을 생성 (3진 분류 모델)
model = BERTCLF(MODEL_NAME, num_claases=3)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7008.11it/s]
[transformers] BertModel LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [98]:
# 평가에서 사용할 함수 선언 
def metrics(eval_pred):
    logits, y = eval_pred
    # logits -> [ x.xxx, x.xxx, x.xxx ]
    pred = logits.argmax(-1)

    return {
        'accuracy_score' : accuracy_score(y, pred), 
        'f1_score' : f1_score(y, pred, average='macro')
    }

In [99]:
# trainer에서 사용할 설정들을 세팅 
args = TrainingArguments(
    output_dir= "/model", 
    eval_strategy= 'epoch', 
    save_strategy='epoch', 
    num_train_epochs=3, 
    learning_rate=5e-5,         # 3e-05, 4e-05, 5e-05 값들을 일반적으로 사용
    weight_decay=0.01, 
    warmup_ratio= 0.1, 
    logging_steps=50, 
    load_best_model_at_end=True, 
    metric_for_best_model='f1_score', 
    greater_is_better=True
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [100]:
trainer = Trainer(
    model = model, 
    args = args, 
    train_dataset= train_tok, 
    eval_dataset= test_ds, 
    compute_metrics= metrics, 
    processing_class= tokenizer             # 구버전에서는 tokenizer 매개변수 
)

In [ ]:
trainer.train()

c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
